# 02 - Train + Evaluate (reproducible)
**Runtime -> Restart runtime -> Run all.** Edit only the parameters cell. Needs a Colab Secret `ROBOFLOW_API_KEY`.
Commit the downloaded `results/` back to the repo afterwards.

In [ ]:
# 1. Parameters -- EDIT ONLY THIS CELL
GITHUB_USER_REPO = "cesaredellacorte-bot/Architectural-items_detection"        # e.g. "team-x/aeco-yolo"
ROBOFLOW_WORKSPACE = "cesare-della-corte"
ROBOFLOW_PROJECT   = "architectural_items_detection_v1"
DATA_SOURCE = "gdrive"                          # "gdrive" (team zip) or "roboflow"
GDRIVE_FILE_ID = "1iVq3qa_xrTHb422iZmxqrK1AS8E2h0La"
ROBOFLOW_VERSION   = 2                          # dataset version (document it in README!)
MODEL      = "yolov8n.pt"                       # yolov8n / yolov8s
EPOCHS, BATCH, IMGSZ, SEED = 50, 16, 640, 0     # assignment requires >= 30 epochs
WEIGHTS_URL = ""                                # GitHub Release link to best.pt (used if FAST_VERIFY)
FAST_VERIFY = False                             # True = 5-epoch run + load real weights (no GPU / short on time)
NEW_IMAGES_DIR = "/content/repo/new_images"     # 5+ images NOT in the dataset (commit them to /new_images)

In [ ]:
# 2. Clone repo + install pinned dependencies
import os, time, sys
T0 = time.time()
if not os.path.exists("/content/repo"):
    !git clone https://github.com/{GITHUB_USER_REPO} /content/repo
%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/scripts")
import evidence
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Dataset (YOLO format, train/valid split kept as provided) + cleaning
import glob, yaml, zipfile
sys.path.insert(0, "/content/repo/scripts")
import prepare_dataset
if DATA_SOURCE == "roboflow":
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    ds = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION).download("yolov8")
    root = ds.location
else:
    import gdown
    os.makedirs("/content/dataset", exist_ok=True)
    gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", "/content/ds.zip", quiet=False)
    zipfile.ZipFile("/content/ds.zip").extractall("/content/dataset")
    root = os.path.dirname(glob.glob("/content/dataset/**/data.yaml", recursive=True)[0])
# Merge the duplicate class 'Furnitures' into 'Furniture' (Walls is kept), convert polygons to boxes, drop duplicates
DATA_YAML, CLASS_NAMES = prepare_dataset.prepare(root, merge={"Furnitures": "Furniture"})
class _DS: location = root
ds = _DS()
print(open(DATA_YAML).read())

In [ ]:
# 4. Train (or short verification run + real weights)
from ultralytics import YOLO
import glob, shutil
model = YOLO(MODEL)
model.train(data=DATA_YAML, epochs=5 if FAST_VERIFY else EPOCHS, batch=BATCH, imgsz=IMGSZ, seed=SEED,
            project="/content/runs", name="train", exist_ok=True, plots=True)
RUN_DIR = str(model.trainer.save_dir)
BEST = f"{RUN_DIR}/weights/best.pt"
if FAST_VERIFY:
    assert WEIGHTS_URL, "Set WEIGHTS_URL to your GitHub Release best.pt"
    !wget -q -O /content/best_real.pt "{WEIGHTS_URL}"
    BEST = "/content/best_real.pt"
print("Using weights:", BEST)

In [ ]:
# 5. Metrics table (P / R / mAP50 / mAP50-95) + curves
best = YOLO(BEST)
rows = evidence.metrics_table(best, DATA_YAML, split="val", imgsz=IMGSZ, out_dir="results")
evidence.copy_curves(RUN_DIR if not FAST_VERIFY else RUN_DIR, "results/curves")
from IPython.display import Image, display
display(Image("results/curves/results.png", width=900))
display(Image("results/curves/confusion_matrix.png", width=600))

In [ ]:
# 6. Evidence pack: 5 annotation examples, 10 validation predictions, 5 new-image predictions
names = best.names
evidence.save_annotation_examples(f"{ds.location}/train/images", f"{ds.location}/train/labels", names, n=5,
                                  out_dir="results/evidence/annotations")
evidence.save_predictions(best, f"{ds.location}/valid/images", 10, "results/evidence/val_predictions", imgsz=IMGSZ)
evidence.save_predictions(best, NEW_IMAGES_DIR, 5, "results/evidence/new_predictions", imgsz=IMGSZ)

In [ ]:
# 7. Error-analysis helper: RED = false positive, BLUE = false negative. Pick 3 FP + 3 FN for docs/error_analysis.md
errs = evidence.find_errors(best, f"{ds.location}/valid/images", f"{ds.location}/valid/labels", names, imgsz=IMGSZ,
                            out_dir="results/evidence/errors")
import glob
from IPython.display import Image, display
for p in sorted(glob.glob("results/evidence/errors/*"))[:6]:
    print(p); display(Image(p, width=700))

In [ ]:
# 8. Reproducibility proof (paste into README)
runtime_min = (time.time() - T0) / 60
evidence.write_run_info("results/run_info.md", extra={
    "Dataset": f"{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT} v{ROBOFLOW_VERSION}", "Model": MODEL,
    "Epochs/Batch/Imgsz": f"{EPOCHS}/{BATCH}/{IMGSZ}", "Mode": "verification run + weights" if FAST_VERIFY else "full training",
    "Total runtime (min)": round(runtime_min, 1)})

In [ ]:
# 9. Download results to commit via GitHub web UI (Add file -> Upload files)
!zip -qr /content/results_to_commit.zip results
from google.colab import files
files.download("/content/results_to_commit.zip")